In [25]:
import json
import pandas as pd
from collections import Counter

# Load JSONL into pandas
records = []
with open("data/all_providers_results_parallelized.jsonl", "r") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("Bad line:", e)
df = pd.DataFrame(records)

print(f"✅ Loaded {len(df)} rows")
print(df.columns.tolist())


✅ Loaded 76 rows
['index', 'question', 'choices', 'correct_answer', 'gemini-2.5-flash-lite_response', 'google_gemma-3n-E4B-it_response', 'openai_gpt-oss-120b_response', 'meta-llama_Llama-3.2-11B-Vision-Instruct_response', 'openai_gpt-oss-20b_response', 'nvidia_NVIDIA-Nemotron-Nano-9B-v2_response', 'accounts_fireworks_models_kimi-k2-thinking_response', 'grok-4-fast-reasoning_response', 'gpt-5-nano-2025-08-07_response']


In [26]:
# Count occurrences of each model key
model_columns = [c for c in df.columns if c.endswith('_response')]
print(f"Models detected ({len(model_columns)} total):")
for m in model_columns:
    print("-", m)



Models detected (9 total):
- gemini-2.5-flash-lite_response
- google_gemma-3n-E4B-it_response
- openai_gpt-oss-120b_response
- meta-llama_Llama-3.2-11B-Vision-Instruct_response
- openai_gpt-oss-20b_response
- nvidia_NVIDIA-Nemotron-Nano-9B-v2_response
- accounts_fireworks_models_kimi-k2-thinking_response
- grok-4-fast-reasoning_response
- gpt-5-nano-2025-08-07_response


In [27]:
counts = {m: df[m].notna().sum() for m in model_columns}
for m, c in counts.items():
    print(f"{m}: {c}")



gemini-2.5-flash-lite_response: 20
google_gemma-3n-E4B-it_response: 20
openai_gpt-oss-120b_response: 20
meta-llama_Llama-3.2-11B-Vision-Instruct_response: 20
openai_gpt-oss-20b_response: 20
nvidia_NVIDIA-Nemotron-Nano-9B-v2_response: 20
accounts_fireworks_models_kimi-k2-thinking_response: 20
grok-4-fast-reasoning_response: 20
gpt-5-nano-2025-08-07_response: 20


In [28]:
import pandas as pd
from IPython.display import display

# Make sure all expected columns exist
for m in model_columns:
    if m not in df.columns:
        df[m] = None

# 1. Group by question index
grouped = df.groupby("index")

# 2. Identify incomplete indices (missing any model response)
incomplete_indices = []
for idx, group in grouped:
    found_models = [m for m in model_columns if group[m].notna().any()]
    if len(found_models) < len(model_columns):
        incomplete_indices.append(idx)

print(f"Total incomplete indices: {len(incomplete_indices)}")
print("First few:", incomplete_indices[:20])

# 3. Gracefully handle empty case
if not incomplete_indices:
    print("\n✅ All indices appear complete after merging.")
else:
    # Pick one incomplete index to inspect
    idx = incomplete_indices[0]
    group = grouped.get_group(idx)
    print(f"\n--- Inspecting index {idx} ---")

    # 4. Merge all fragments (later rows overwrite earlier)
    merged = group.iloc[0].copy()
    for m in model_columns:
        vals = group[m].dropna().tolist()
        if vals:
            merged[m] = vals[-1]  # take most recent non-null

    # 5. Display all responses neatly
    cols_to_show = ["index", "question", "correct_answer"] + model_columns
    pd.set_option("display.max_colwidth", 200)
    display(merged[cols_to_show].to_frame().T)

Total incomplete indices: 0
First few: []

✅ All indices appear complete after merging.


In [29]:
import json
import pandas as pd
from IPython.display import display, HTML

# Load JSONL
records = []
with open("data/all_providers_results_parallelized.jsonl") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

df = pd.DataFrame(records)

# Group by question index
grouped = df.groupby("index")

def show_group(idx, group):
    """Render one question group neatly."""
    model_cols = [c for c in group.columns if c.endswith("_response")]
    # Also check for reasoning columns (ending with _reasoning or _thinking)
    reasoning_cols = [c for c in group.columns if c.endswith("_reasoning") or c.endswith("_thinking")]
    q = group.iloc[0]  # question text and choices (same within group)

    html = f"""
    <h3>Question index: {idx}</h3>
    <p><b>Question:</b> {q['question']}</p>
    <p><b>Choices:</b> {q['choices']}</p>
    <p><b>Correct answer:</b> {q['correct_answer']}</p>
    """

    rows = []
    for _, row in group.iterrows():
        for m in model_cols:
            if m in row and pd.notna(row[m]):
                val = row[m]
                if isinstance(val, dict):
                    answer = val.get("answer", str(val))
                    # Check for reasoning/thinking fields in the response dict
                    reasoning = val.get("thinking") or val.get("reasoning") or val.get("reasoning_text", "N/A")
                    if reasoning is None or reasoning == "":
                        reasoning = "N/A"
                    elif isinstance(reasoning, str) and len(reasoning) > 200:
                        reasoning = reasoning[:200] + "..."
                    
                    input_tokens = val.get("input_tokens", "N/A")
                    output_tokens = val.get("output_tokens", "N/A")
                    first_token_latency = val.get("first_token_latency", "N/A")
                    total_latency = val.get("total_latency", "N/A")
                    
                    # Format latency values
                    if isinstance(first_token_latency, (int, float)):
                        first_token_latency = f"{first_token_latency:.3f}s"
                    if isinstance(total_latency, (int, float)):
                        total_latency = f"{total_latency:.3f}s"
                    elif total_latency is None:
                        total_latency = "N/A"
                else:
                    answer = str(val)
                    reasoning = "N/A"
                    input_tokens = "N/A"
                    output_tokens = "N/A"
                    first_token_latency = "N/A"
                    total_latency = "N/A"
                
                rows.append({
                    "Model": m.replace("_response", ""),
                    "Answer": answer,
                    "Reasoning": reasoning,
                    "Input Tokens": input_tokens,
                    "Output Tokens": output_tokens,
                    "First Token Latency": first_token_latency,
                    "Total Latency": total_latency
                })
    
    # Also check for separate reasoning columns
    for _, row in group.iterrows():
        for r_col in reasoning_cols:
            if r_col in row and pd.notna(row[r_col]):
                # Find corresponding model name
                model_name = r_col.replace("_reasoning", "").replace("_thinking", "")
                # Check if we already have this model in rows
                found = False
                for i, r in enumerate(rows):
                    if r["Model"] == model_name or model_name in r["Model"]:
                        if r["Reasoning"] == "N/A":
                            reasoning_val = str(row[r_col])
                            if len(reasoning_val) > 200:
                                reasoning_val = reasoning_val[:200] + "..."
                            rows[i]["Reasoning"] = reasoning_val
                        found = True
                        break
                if not found:
                    reasoning_val = str(row[r_col])
                    if len(reasoning_val) > 200:
                        reasoning_val = reasoning_val[:200] + "..."
                    rows.append({
                        "Model": model_name,
                        "Answer": "N/A",
                        "Reasoning": reasoning_val,
                        "Input Tokens": "N/A",
                        "Output Tokens": "N/A",
                        "First Token Latency": "N/A",
                        "Total Latency": "N/A"
                    })
    
    table = pd.DataFrame(rows)

    html += table.to_html(index=False, escape=False)
    html += "<hr>"
    display(HTML(html))

# Display first 10 unique indices (grouped)
for i, (idx, group) in enumerate(grouped):
    show_group(idx, group)
    if i == 9:
        break


Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Explanation/Reasoning: The passage states that 2.6 percent of *healthy* women had depressed partners, not that 2.6 percent of healthy women *were depressed*. This is a crucial distinction. Therefore, statement 1 is not true according to the text.\nFinal Answer: 1",N/A,569,62,0.469s,0.569s
google_gemma-3n-E4B-it,"Okay, let's break down this question. We need to find the statement that is *not* supported by the passage.\n\n* **Statement 0:** ""Ten percent of women who were depressed had depressed partners."" The passage explicitly states, ""Ten percent of women who were depressed had depressed partners."" So, this is true.\n\n* **Statement 1:** ""2.6 percent of healthy women were depressed."" The passage says, ""For the healthy women, the figure was only 2.6 percent."" So, this is also true.\n\n* **Statement 2:** ""Special attention should be paid to families in which both the father and the mother were depressed."" The passage mentions that ""Previous research suggests that families with two depressed parents may need special attention."" So, this is true.\n\n* **Statement 3:** ""Primary school children whose parents were both depressed couldn't get along well with their peers."" The passage quotes a researcher saying, ""primary school children with two troubled parents have difficulty relating to their peers."" So, this is true.\n\nIt seems I've made a mistake in my previous reasoning. All the statements *are* supported by the passage. Let me re-read carefully. Ah, the wording of choice 1 is a bit misleading. It doesn't say that *only* 2.6% of healthy women were depressed, it says that *among* healthy women, the percentage of those with depressed partners was 2.6%. This is still indirectly supported, but not as directly as the other statements.\n\nLet me think... I need to find the statement that is *least* directly supported or potentially misinterpreted. The statement about healthy women is the most ambiguous.\n\nFinal Answer: 1",N/A,577,364,0.443s,7.914s
openai_gpt-oss-120b,Explanation/Reasoning: The passage states that 10 % of depressed women had depressed partners and that for healthy women the figure was 2.6 % (meaning 2.6 % of healthy women had depressed partners). It does **not** say that 2.6 % of healthy women themselves were depressed; that misrepresents the statistic. All other statements accurately reflect the passage.\n\nFinal Answer: 1,"We need to select which statement is NOT true according to passage. Let's examine each.\n\nPassage says: ""Ten percent of women who were depressed had depressed partners."" That's true. So choice 0 is tru...",582,411,6.843s,7.726s
meta-llama_Llama-3.2-11B-Vision-Instruct,"Explanation/Reasoning: The passage actually states that 10% of women who were depressed had depressed partners, not the other way around. It says ""Ten percent of women who were depressed had depressed partners,"" but it's the partners who were depressed, not the women themselves.\n\nFinal Answer: 0",N/A,571,62,0.703s,6.442s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The passage states that 2.6 percent of healthy women had depressed partners, not that 2.6 percent of healthy women themselves were depressed. The 13% figure refers to partners of depressed mothers, not the women's own depression rates. The other options (0, 2, 3) are directly supported by the text. \n\nFinal Answer: 1","Okay, let's tackle this question. The task is to find which statement is NOT true based on the passage. Let me go through each option one by one.\n\nStarting with choice 0: ""Ten percent of women who wer...",565,418,7.619s,8.397s
openai_gpt-oss-20b,"Explanation/Reasoning: \nThe passage states that *10% of depressed women had depressed partners*, whereas for healthy women only *2.6% had depressed partners*. It does **not** say that 2.6% of healthy women themselves were depressed. Therefore statement 1 is false, whil

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Explanation/Reasoning: The passage strongly suggests that Kauai is not for those who prefer city amenities, luxury hotels, or extensive shopping. It highlights the island's natural beauty, unpredictable weather (which can be seen as a negative by some), lack of typical tourist attractions like discos and four-star restaurants, and emphasis on ""going natural."" The author repeatedly discourages people who *don't* appreciate these aspects from visiting. Conversely, the very things the author presents as drawbacks – the unpredictable weather, the focus on nature, the limited shopping, the slow pace, and the ""do-it-yourself"" activities – are exactly what someone who loves nature would seek out. The author even points out that ""Hawaiians, surfers, New Agers, and people who love nature and beauty and want a different quality of life"" are the ones who appreciate Kauai.\nFinal Answer: 0",N/A,690,182,0.462s,1.190s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage heavily emphasizes the unpredictable weather, lack of luxury amenities like fine dining and nightlife, and limited shopping options on Kauai. It directly contrasts Kauai with islands like Maui, Lanai, and the Big Island, which are presented as better choices for those seeking comfort and convenience. The author explicitly states that Kauai is ""not about pampering"" and is geared towards those who want a ""do-it-yourself"" experience focused on nature. Therefore, the people who would *not* like Kauai are those who love the comfort in a fine hotel.\n\nFinal Answer: 2",N/A,698,122,0.495s,3.732s
openai_gpt-oss-120b,"Explanation/Reasoning: The passage repeatedly emphasizes that Kauai is best suited for people who enjoy natural, outdoor experiences—walking along the coast, diving, swimming, and connecting with nature. It discourages those looking for fine hotels, city amenities, shopping, or comfort. Therefore, the group that would like to visit Kauai are those who love nature.\n\nFinal Answer: 0","We need to determine which group would like to visit Kauai based on the text. The passage is discouraging most people, but says ""Kauai is not about pampering. It's about going natural and finding the ...",690,202,2.988s,4.805s
openai_gpt-oss-20b,"Explanation/Reasoning: The passage is a strong warning against visiting Kauai, highlighting its unpredictable weather, lack of hotels, restaurants, shopping, and nightlife. It repeatedly emphasizes that Kauai is a place for “going natural” and “finding the nature within you.” The only audience whose interests match these qualities are those who love nature. Therefore, the correct answer is the group that loves nature. \n\nFinal Answer: 0","We need to answer: ""Who would like to visit Kauai?"" According to the passage, the preceding content is urging people not to go to Kauai, except those who love nature. The passage has negative tone tow...",690,322,4.927s,6.538s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The passage emphasizes Kauai's natural beauty, outdoor activities, and connection to nature, while criticizing its lack of modern comforts like hotels, shopping, and city amenities. It explicitly states Kauai is for those who want to ""go natural"" and embrace a different lifestyle, which aligns with nature lovers. Choices 1, 2, and 3 are directly contradicted by the text, as Kauai lacks city life, hotel comfort, and shopping opportunities. \n\nFinal Answer: 0","Okay, let's tackle this question. The user is asking who would like to visit Kauai based on the passage provided. The choices are 0 to 3, each representing different types of people.\n\nFirst, I need to...",693,473,7.317s,8.607s
grok-4-fast-reasoning,"The passage is basically a sarcastic love letter to Kauai, pushing away tourists who want luxury hotels, shopping, discos, or predictable weather, while hyping up its wild natural beauty, rainbows, beach

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Explanation/Reasoning: The passage states that Heather Jack and her family prepared a dinner for an elderly woman and her son, and then headed home to prepare for a visit from Santa. This implies they were preparing for their own Christmas festivities. The question asks what they *didn't* do. They didn't prepare a dinner for a *poor family* specifically, but rather for an elderly woman and her son. They did chat with them, and they did make preparations for Santa's visit. Visiting a good friend isn't mentioned at all. However, looking at the options, option 0 states ""preparing a dinner for a poor family."" While the family they helped might have been poor, the description focuses on the woman's age and her son's disability, not necessarily their poverty. Option 3, ""visiting one of their good friends in other district,"" is not supported by the text at all. The most accurate answer for what they *didn't* do is visit a good friend in another district, as this is completely absent from the narrative.\n\nHowever, upon closer re-reading, the question asks what the Heathers did *except* last Christmas Eve. This implies that the options are things that could have happened on Christmas Eve, and we're looking for the one that *didn't* happen.\n\nLet's re-evaluate:\n0: ""preparing a dinner for a poor family"" - The text says they prepared dinner for an elderly woman and her son. While not explicitly stated as ""poor,"" this is a plausible interpretation and they *did* prepare dinner.\n1: ""chatting with the elderly mother and her disabled son"" - This is explicitly stated in the text.\n2: ""making preparations for their own Christmas festival"" - The text says they headed home ""to prepare for a visit from Santa."" This counts as preparing for their own festival.\n3: ""visiting one of their good friends in other district"" - This is not mentioned anywhere in the text.\n\nTherefore, the activity that the Heathers did *not* do is visit one of their good friends in another district.\n\nFinal Answer: 3",N/A,538,448,0.454s,1.832s
google_gemma-3n-E4B-it,"Okay, let's break this down. The question asks what Heather Jack *didn't* do last Christmas Eve. The passage specifically states that she and her family prepared a dinner for an elderly woman and her son, stayed to chat, and then went home to prepare for Santa's visit. \n\nNow let's look at the options:\n\n* **0: preparing a dinner for a poor family** - The passage says they prepared a dinner for an elderly woman, not a poor family.\n* **1: chatting with the elderly mother and her disabled son** - The passage explicitly states they chatted for an hour.\n* **2: making preparations for their own Christmas festival** - The passage says they went home to prepare for a visit from Santa, which implies preparing for their own Christmas celebration.\n* **3: visiting one of their good friends in other district** - There's no mention of visiting friends in another district.\n\nSince the question asks what Heather *didn't* do, and option 0 is the only thing she didn't do according to the text, that's the correct answer.\n\nFinal Answer: 0",N/A,546,245,0.498s,6.550s
openai_gpt-oss-20b,"Explanation/Reasoning: Heather and her family prepared a dinner for an elderly woman and her son, chatted with them, and then headed home to get ready for Santa. There's no mention of them visiting a friend, so that is the action they did not do. \nFinal Answer: 3","We need to answer ""The Heathers did all the following things except_last Christmas Eve."" The choices are:\n\n0: preparing a dinner for a poor family\n1: chatting with the elderly mother and her disabled ...",548,349,6.663s,7.385s
openai_gpt-oss-120b,"Explanation/Reasoning: According to the passage, on last Christmas Eve Heather and her family went to a neighbor’s house, prepared a dinner for an elderly woman and her son, and chatted with them before he

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Well, it sounds like Toby is really hoping to find a specific color for his ""Super Racer"" after it was stolen. He mentions that the toy company only has red, blue, and green, but then his dad found out Cobb's Toy Store *might* have purple ones. Toby specifically asks if they can go there when he visits his grandpa, which strongly suggests that's the color he's after.\n\nFinal Answer: 3",N/A,359,91,0.450s,0.718s
google_gemma-3n-E4B-it,"Explanation/Reasoning: Toby is asking if they can go to Cobb's Toy Store to get a purple Super Racer because the one that was stolen was purple, and the company only has red, blue, and green ones. So, he wants a purple racer.\nFinal Answer: 3",N/A,367,61,0.584s,2.415s
openai_gpt-oss-120b,"Explanation/Reasoning: In the letter, Toby says the toy company only offers red, blue, and green racers, but his dad found out that Cobb's Toy Store might have purple ones. Toby asks if they can go to that store, indicating he wants the purple racer.\n\nFinal Answer: 3","We need to read the question. It is a passage (a letter). The writer (Toby) says: ""I was so upset last week when somebody stole my Super Racer from the park by your house. I called up the toy company ...",378,250,4.312s,5.552s
openai_gpt-oss-20b,"Explanation/Reasoning: The letter states that Dad called Cobb's Toy Store and discovered they might have purple racers. Toby asks if they can go there to get one, implying she wants a purple racer. Thus the missing word is ""purple,"" which corresponds to choice index 3.\n\nFinal Answer: 3","We need to parse the scenario. The writer Toby wants to have a ___ racer. They mention ""Dad called Cobb's Toy Store today and found out they might have purple ones."" So Toby wants to have a purple rac...",378,363,5.991s,7.125s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: In the letter, Toby mentions that the toy company only has red, blue, and green racers, but Cobb's Toy Store might have purple ones. Since Toby is asking to visit Cobb's to get a new racer, the desired color is purple. \nFinal Answer: 3","Okay, let's see. The question is about what color racer Toby wants. The letter from Toby says that someone stole his Super Racer. He called the toy company, but they only have red, blue, and green. Th...",364,276,5.816s,6.985s
meta-llama_Llama-3.2-11B-Vision-Instruct,"It seems that the writer is looking for a specific color of the Super Racer toy. They mention that the toy company only has red, blue, and green racers, but then they mention that Cobb's Toy Store might have purple ones. This suggests that the writer is particularly interested in having a purple Super Racer.\n\nExplanation/Reasoning: The writer mentions that Cobb's Toy Store might have purple racers, which is the only color not mentioned as being available from the toy company. This implies that the writer is interested in having a purple Super Racer, as they are inquiring about it specifically.\nFinal Answer: 3",N/A,369,129,0.432s,10.070s
grok-4-fast-reasoning,"The letter mentions that Toby's Super Racer was stolen, and the toy company only offers red, blue, or green ones. But he's excited about the possibility of getting a purple racer from Cobb's Toy Store, and he's specifically asking to go there during his visit. So, it's clear he wants a purple one to replace the stolen toy.\n\nExplanation/Reasoning: Toby's stolen racer isn't specified by color, but he inquires about purple ones available at the store, showing that's what he desires.\nFinal Answer: 3",N/A,495,397,12.461s,12.698s
accounts_fireworks_models_kimi-k2-thinking,"Explanation/Reasoning: Toby mentions that his dad called Cobb's Toy Store and learned they might have purple racers, and he asks if they can go there when he visits. This shows he wants a purple racer.\nFinal Answer: 3","We need to analyze the question. The question is: ""The writer wants to have a ___ ra

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Well, according to the story, the fox told Mother Bear to put her tail in the hole in the ice for a long time, hoping she'd catch fish. So, in essence, he was teaching her how to fish, even if his method was a bit tricky!\n\nFinal Answer: 3",N/A,385,62,0.437s,0.477s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The story explicitly states that the fox told Mother Bear how to fish. He gave her instructions on how to use her tail to catch fish. So, the answer is 3.\n\nFinal Answer: 3",N/A,393,49,0.567s,1.690s
openai_gpt-oss-120b,"Explanation/Reasoning: In the story, the fox promises Mother Bear that he will teach her how to fish and instructs her to put her tail in the icy water. This corresponds to the choice that says the fox taught her how to fish.\n\nFinal Answer: 3","We need to read the question: ""What did a fox tell Mother Bear to do on a cold winter day?"" The story: the fox told Mother Bear that he would teach her how to fish; he took her to a hole in the ice, a...",404,300,5.288s,6.146s
meta-llama_Llama-3.2-11B-Vision-Instruct,"It sounds like a trickster fox story! The fox told Mother Bear to put her tail down into the water and keep it there for a long time, and then she would find fish around her tail when she pulled it out. It's a pretty sneaky trick, and it's no wonder Mother Bear fell for it! She ended up with a part of her tail frozen in the ice, and that's why her tail is shorter than usual.\n\nFinal Answer: 3",N/A,391,96,0.417s,8.240s
openai_gpt-oss-20b,Explanation/Reasoning: The story says the fox claimed he would teach Mother Bear how to fish and gave her a trick involving her tail in the ice. This matches the choice that he “taught her how to fish.” \nFinal Answer: 3,"We need to interpret the question: ""What did a fox tell Mother Bear to do on a cold winter day?"" The given narrative: fox told Mother Bear he would teach her how to fish. He told her to put her tail i...",404,321,6.016s,6.906s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The story clearly states that the fox wanted to teach Mother Bear how to fish. He instructed her to put her tail in the water to catch fish, which aligns with option 3. The other choices (swimming, playing, or catching fish for him) are not supported by the narrative. The key detail is the fox's intent to teach her a fishing method. \nFinal Answer: 3","Okay, let's see. The question is asking what the fox told Mother Bear to do on a cold winter day. The story goes that the fox wanted to teach her how to fish. He took her to a hole in the ice and told...",380,273,4.560s,6.103s
accounts_fireworks_models_kimi-k2-thinking,"Explanation/Reasoning: The passage says the fox told Mother Bear he would teach her how to fish, and he instructed her to put her tail in the water and keep it there. This matches the description that he taught her how to fish. \nFinal Answer: 3","We have a multiple-choice question. The question: ""What did a fox tell Mother Bear to do on a cold winter day?""\n\nFirst, we need to carefully read the passage.\n\nThe passage: On a cold winter day, a fox...",375,1112,17.349s,18.194s
grok-4-fast-reasoning,"Explanation/Reasoning: The story describes the fox telling Mother Bear that he would teach her how to fish by having her put her tail in the ice hole to catch fish, which was actually a trick. Among the choices, this matches option 3, as that's what the fox claimed to do and instructed her on.\n\nFinal Answer: 3",N/A,520,539,12.789s,12.882s
gpt-5-nano-2025-08-07,"Explanation/Reasoning: The fox said he would teach her how to fish and instructed her to put her tail in the water and wait for fish, effectively teaching her to fish. The correct choice is 3.\n\nFinal Answer: 3",N/A,374,630,18.100s,18.723s


Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Jean Clottes is quoted as saying, ""People think that because the pictures have been there so long they will always continue to be there."" This directly implies that visitors believe the drawings are so enduring that they won't disappear.\n\nFinal Answer: 3",N/A,909,52,0.586s,0.616s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage states that Jean Clottes says, ""People think that because the pictures have been there so long they will always continue to be there."" This implies that people *do* believe the drawings are old, but that doesn't directly address what they *think* about the drawings themselves. The passage doesn't suggest they think they are allowed to paint there (choice 1), or that they think the drawings will not disappear (choice 3). However, Clottes' statement highlights a problem: people underestimate the fragility of the drawings because of their age. The core issue is that people don't fully appreciate the value and vulnerability of the ancient art. Therefore, the most accurate inference is that people think the drawings will not disappear because of their age, but don't realize how quickly they are being destroyed. The best option that reflects this is choice 2: think the drawings should be left alone. While not explicitly stated, the underlying reason for the damage is a lack of respect for the artwork, stemming from a belief in its permanence.\n\nFinal Answer: 2",N/A,917,226,0.542s,5.800s
openai_gpt-oss-120b,"Explanation/Reasoning: In the passage, Jean Clottes notes that visitors assume the rock drawings will persist indefinitely because they have been there so long, implying they think the drawings will not disappear. This matches choice 3.\n\nFinal Answer: 3","We need to understand the passage. The last question: ""Jean Clottes says that people who visit the mountain _ .""\n\nWe need to find the statement that matches something Jean Clottes says about visitor...",909,399,6.836s,7.445s
meta-llama_Llama-3.2-11B-Vision-Instruct,"It seems like the main issue here is that tourists are not respecting the ancient drawings on the rocks at Mont Bego in the French Alps. They're damaging the artwork by stamping on it, wearing away the rock, and even cutting off parts to take home as souvenirs. Some visitors are also rubbing the drawings to get a clearer picture, which is further damaging the artwork.\n\nIt's not that people don't think the drawings are old, or that they believe they're allowed to paint there. It's more that many tourists seem to be unaware of or don't care about the historical significance of the site, and are treating it like a normal tourist destination.\n\nThe experts are divided over the best way to preserve the drawings, with some suggesting that the site be closed off to the public except for guided tours, while others think that's too extreme and that a more feasible solution would be to charge a fee or provide guides to educate visitors about the importance of preserving the site.\n\nSo, considering the options provided, it seems like the most accurate description of what people who visit the mountain do is that they think the drawings should be left alone.\n\nFinal Answer: 2",N/A,905,236,1.941s,8.454s
openai_gpt-oss-20b,Explanation/Reasoning: In the passage Jean Clottes remarks that visitors mistakenly believe the ancient drawings will remain unchanged simply because they have existed for thousands of years. This directly matches option 3: “think the drawings will not disappear.” The other options do not reflect his stated beliefs about how visitors perceive the artwork.\n\nFinal Answer: 3,"We need to infer from passage: ""Jean Clottes says that people who visit the mountain _____"". Let's see context: Clottes criticized tourists stamping, cutting, rubbing, leaving graffiti; He says they t...",909,379,6.169s,7.161s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: 

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"This is a really touching story! The text focuses on how love and patient training helped a traumatized dog named Jessie overcome his fear and aggression. While there's a conflict mentioned, it's not the main theme. The core message is about the transformation of Jessie through love, which ultimately calmed his inner ""storm."" Therefore, ""Love Calmed the Storm"" seems to best capture the essence of the narrative.\n\nFinal Answer: 1",N/A,581,90,0.579s,0.995s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The story is primarily about how love and patience transformed a traumatized dog named Jessie. While there are conflicts, the central theme is the power of love to heal and change a dog's behavior. ""Love Calmed the Storm"" (option 1) captures this essence really well. ""A Dog Named Jessie"" (option 0) is too simple; it just states a fact about the story, not the core message. ""Conflicts Between Dogs"" (option 2) is too narrow; the story is about more than just fights. And ""Fights Between Man and Dog"" (option 3) only focuses on a small part of the narrative. Therefore, ""Love Calmed the Storm"" is the most fitting and comprehensive title.\n\nFinal Answer: 1",N/A,589,163,0.543s,4.578s
openai_gpt-oss-120b,"Explanation/Reasoning: The passage emphasizes how the husband’s love and patient training helped calm the frightened, aggressive dog Jessie, transforming his behavior. The most fitting title reflects this central theme of love soothing the turmoil, which is captured by “Love Calmed the Storm.” The other options focus merely on the dog’s name or on conflict without highlighting the transformative love.\n\nFinal Answer: 1","We need to determine best title for the text. The passage describes love and patience calming a traumatized dog, describing transformation and love. Titles: ""A Dog Named Jessie"" (just name), ""Love Cal...",598,175,2.039s,3.673s
openai_gpt-oss-20b,"Explanation/Reasoning: The passage emphasizes how love and gentle training turned a traumatized, fearful dog into a calm and loyal companion, illustrating the healing power of affection. The title that best captures this theme is ""Love Calmed the Storm,"" which reflects the transformation and soothing effect of love described in the text. \nFinal Answer: 1","We need to pick best title. Text focuses on love calming storm, devotion, transformation. Option 1: Love Calmed the Storm. That fits content. Option 0: A Dog Named Jessie is generic. Option 2: Conflic...",598,155,1.426s,2.861s
meta-llama_Llama-3.2-11B-Vision-Instruct,"The title that best captures the essence of the story is ""Love Calmed the Storm"". The text is not just about a dog named Jessie, but about the transformative power of love and affirmation in overcoming trauma and fear. The story highlights how Jessie, a previously traumatized dog, is able to overcome his fears and become a loving companion through the gentle persuasion and love of his owner. The phrase ""Love Calmed the Storm"" is a perfect summary of the story's theme, which is not just about the conflicts between dogs, but about the redemptive power of love in healing emotional wounds.\n\nFinal Answer: 1",N/A,591,126,4.375s,7.960s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The text emphasizes how love and patience from the husband transformed Jessie, who was once traumatized and aggressive. The phrase ""affirmation and love had calmed the storm"" directly ties to the theme of love overcoming past struggles. While conflicts and fights are part of the story, they serve to illustrate the central message of love's transformative power. ""A Dog Named Jessie"" is too generic, and the other options focus on specific events rather than the core theme. \nFinal Answer: 1","Okay, let's tackle this question. The user wants the best title for the text provided. The story is about Jessie, a dog who was traumatized and had a conflic

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Explanation/Reasoning: The passage states that ""The short-story reader can expect to find certain basic elements in any story. For example, all stories involve a person or persons, in a particular setting, faced with a demand for a response."" It then goes on to explain how these elements, along with others like structure, mood, and tone, are used to imply meanings that the reader infers. Therefore, inferring in fiction is based on these fundamental elements that make up a story.\nFinal Answer: 1",N/A,569,107,0.414s,0.766s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage emphasizes that inferring in fiction is based on understanding the basic elements of the story – action, character, setting, structure, mood, tone, and point of view – and how they all connect. The author states that the author provides a ""network of points which serve as clues to his meaning."" These clues aren't direct statements, but elements that the reader must interpret and connect to infer the story's significance or theme. While the other options might be *part* of the process, inferring isn't *based* on just guessing (0), the setting alone (2), or the relationships between people (3). It's the combination and interplay of *all* these elements that allow for inference.\n\nFinal Answer: 1",N/A,577,161,0.507s,4.756s
openai_gpt-oss-120b,"Explanation/Reasoning: The passage explains that to infer the significance of a story, a reader must understand all the facts presented and how they relate to each other. These “facts” are essentially the basic elements of the story (characters, setting, action, etc.) and their interrelationships. Therefore, inference in fiction is based on the basic elements of the story. \n\nFinal Answer: 1","We need to determine correct answer based on passage. Question: ""What is inferring in fiction based on?"" The passage says: ""When the reader of a story understands all the facts and their interrelation...",593,597,8.695s,9.335s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The passage emphasizes that inferring in fiction relies on understanding the interrelationships between all elements of the story, including characters, setting, action, and theme. While basic elements (choice 1) are foundational, the act of inference specifically depends on how these elements connect and interact to convey deeper meaning. The example of the child crying and the observer inferring hurt illustrates how actions and relationships imply unspoken truths. Choice 3 directly addresses this focus on interrelationships, making it the most accurate answer. \nFinal Answer: 3","Okay, let's tackle this question. The user is asking what inferring in fiction is based on, with four choices provided.\n\nFirst, I need to recall the passage. The main idea is that inference in fiction...",574,488,7.905s,8.888s
meta-llama_Llama-3.2-11B-Vision-Instruct,"To infer in fiction is based on the basic elements of the story. These elements, such as action, character, and setting, provide the foundation for the author to convey their meaning. By understanding the relationships between these elements, the reader can piece together the clues provided by the author to infer the significance of the story as a whole. It's not just about guessing or focusing on the setting, but rather about recognizing how all the various elements work together to create a cohesive message.\n\nFinal Answer: 1",N/A,581,102,0.615s,11.575s
openai_gpt-oss-20b,"Explanation/Reasoning:\nThe passage explains that inferring meaning in fiction requires understanding all the facts of the story—its characters, setting, action, motives, etc.—and how these facts interrelate. These facts constitute the basic elements of the story, and inference is built upon them. Therefore, inferring in fiction is based on the basic elements of the story.\n\nFinal Answer: 1","We need to parse questio

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Wow, that's an incredibly moving story about your father. It really highlights his compassion and integrity. Based on everything you've described, the most fitting description for his father in the eyes of the author is undeniably helpful. He went above and beyond to assist someone in need when others wouldn't, demonstrating true character.\n\nFinal Answer: 3",N/A,514,71,0.405s,0.653s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage strongly emphasizes the father's proactive and compassionate actions. He didn't ignore the situation like his colleagues. Instead, he took the initiative to investigate, understand what was happening, and then dedicate his time and effort to helping the man get the care and financial security he needed. This clearly demonstrates a helpful and caring nature. While the father might have instilled strong values in his son, the core action described is one of assistance and support, not strictness or hopefulness. Stubbornness isn't really reflected in the narrative; he was open to understanding and assisting.\n\nFinal Answer: 3",N/A,522,129,0.626s,4.388s
openai_gpt-oss-120b,"Explanation/Reasoning: The passage emphasizes the father's compassionate actions—taking the man to the hospital, arranging a caretaker, handling his finances, and ensuring his wellbeing. These deeds illustrate that the author views his father as someone who is very helpful and caring toward others in need.\n\nFinal Answer: 3","We have a passage describing father helping a man with early Alzheimer’s, taking care, etc. The author says his father took personal time to help another man that many forgot; his character is shown. ...",528,164,2.163s,3.257s
openai_gpt-oss-20b,"Explanation/Reasoning: \nThe passage emphasizes the father’s compassionate actions: he noticed a colleague’s deteriorating condition, advocated for medical care, arranged financial help, and prevented exploitation. These deeds highlight his willingness to help others in need, rather than strictness, stubbornness, or merely hopefulness. Therefore, the author portrays the father as helpful. \n\nFinal Answer: 3","We need to determine correct answer. The passage describes father’s character. They say he had strong values, helped a man neglect. Father is helpful, compassionate, caring, altruistic. Among choices:...",528,170,1.339s,3.224s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The father's actions in the story clearly demonstrate his willingness to assist someone in need, especially when others ignored the situation. He took proactive steps to address the man's health, financial, and social issues, which aligns with being described as ""helpful."" The other options—hopeful, strict, or stubborn—do not accurately reflect the nature of his actions or the author's perspective. \nFinal Answer: 3","Okay, let's see. The question is about determining the author's view of their father based on the story provided. The father helped a man with Alzheimer's who was being taken advantage of. The options...",510,315,5.905s,7.197s
accounts_fireworks_models_kimi-k2-thinking,"Explanation/Reasoning: The passage describes how the author's father noticed a coworker showing signs of Alzheimer's disease and took extensive personal action to help him—arranging medical care, securing a caretaker, and protecting his finances from exploitation. The author explicitly states this ""truly shows his character"" and emphasizes that the father took time to help someone others ignored or neglected. While the father was determined, ""stubborn"" doesn't capture the positive, compassionate nature of his actions. ""Hopeful"" and ""strict"" are not supported by the text. The central theme is the father's willingness to actively assist someone in need, making ""helpful"" the most accurate description.\n\nFinal Answer: 3",The user is asking me to answer a multiple-choice question

Model,Answer,Reasoning,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"The passage describes several inventions that didn't become widely popular. Let's break down why each choice is or isn't supported by the text:\n\n* **0: The father used his invention to stop children watching too much TV.** The passage says the scientist *thought* his children watched too much TV and *connected the TV to an exercise bike* so that pedaling produced the electricity. This implies the *purpose* was to discourage excessive TV watching by making it physically demanding, so this statement is correct.\n\n* **1: It was very bad for the drivers to blow into a bag before their driving.** The passage states that drivers *might need to drive in an emergency even if they had drunk too much alcohol*, which was the *reason* people didn't like the invention. It doesn't say blowing into the bag itself was ""very bad"" for them, just that the restriction was problematic for emergencies.\n\n* **2: The bike crossing rivers was considered one of the best inventions.** The author explicitly calls the bicycle that crosses rivers ""one of the strangest inventions"" and questions why one wouldn't use a bridge instead. This contradicts the idea that it was considered one of the best.\n\n* **3: The invention of new shoes would make players run much faster.** The rotating pad of studs was intended to help change direction more quickly, not necessarily to make players run faster. The passage also highlights the *problem* with the shoes – they made you change direction constantly, which is likely not conducive to speed.\n\nTherefore, the only statement directly supported by the passage is that the father used his invention to stop children from watching too much TV.\n\nFinal Answer: 0",N/A,703,352,0.415s,1.665s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage describes several inventions, some more practical than others, that didn't gain widespread popularity. The question asks what we can infer from the passage. Option 0 is incorrect because the passage states the father thought his children watched *too much* TV and created the bike to *reduce* that, not to stop it entirely. Option 1 is incorrect; the breathalyzer device was designed to *prevent* people from driving under the influence. Option 2 accurately reflects the passage – the bicycle designed to cross rivers is presented as one of the *strangest* inventions, implying it wasn't considered a good or practical idea and is therefore not one of the ""best."" Option 3 is incorrect because the rotating studs in the football shoes actually caused problems with uncontrolled directional changes, hindering speed and agility. Therefore, the most accurate inference is that the river-crossing bicycle wasn't a successful invention.\nFinal Answer: 2",N/A,711,197,0.604s,4.788s
openai_gpt-oss-120b,Explanation/Reasoning: The passage describes a scientist (referred to as a father or parent) who created an invention linking a TV to an exercise bike to limit his children’s TV watching. This directly supports statement 0. The other statements are not supported by the passage’s details.\n\nFinal Answer: 0,"We need to read passage and determine which statement can be known (i.e., true according to passage). Options:\n\n0: The father used his invention to stop children watching too much TV. The passage ment...",715,419,7.460s,8.062s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The passage states that a scientist connected the TV to an exercise bike to make children pedal to watch TV, which aligns with choice 0. Choice 1 is incorrect because the invention aimed to prevent drunk driving, not make it worse. Choice 2 is wrong as the river-crossing bike was described as strange, not one of the best. Choice 3 is false because the shoes focused on direction changes, not speed. \nFinal Answer: 0","Okay, let's tackle this question. The user wants to know which statement is correct based 

In [30]:
import pandas as pd
import numpy as np
from IPython.display import display

# Make sure all model columns exist
for m in model_columns:
    if m not in df.columns:
        df[m] = np.nan

# Check for reasoning columns (ending with _reasoning or _thinking)
reasoning_columns = [c for c in df.columns if c.endswith("_reasoning") or c.endswith("_thinking")]
print(f"Found {len(reasoning_columns)} reasoning columns: {reasoning_columns}")

# Helper to detect errors in strings or dicts
def is_error(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        val_str = value.strip()
        # Check for "Error:" prefix or error-like patterns
        if val_str.startswith("Error:") or "Error" in val_str or "ConnectionResetError" in val_str:
            return True
    elif isinstance(value, dict):
        # Check if dict has an error field or error in answer
        if "error" in value or (isinstance(value.get("answer"), str) and "Error" in value.get("answer", "")):
            return True
    return False

# Helper to check if reasoning exists in response dict
def has_reasoning(response_val):
    """Check if reasoning/thinking exists in response."""
    if pd.isna(response_val):
        return False
    if isinstance(response_val, dict):
        reasoning = response_val.get("thinking") or response_val.get("reasoning") or response_val.get("reasoning_text")
        return reasoning is not None and reasoning != "" and str(reasoning).strip() != ""
    return False

# Group by index and merge responses from different rows
grouped = df.groupby("index")
issues = []
reasoning_stats = []

for idx, group in grouped:
    # Merge all model responses for this index (take last non-null value)
    merged_row = group.iloc[0].copy()
    for m in model_columns:
        vals = group[m].dropna().tolist()
        if vals:
            merged_row[m] = vals[-1]  # take most recent non-null
    
    # Check each model for missing/error responses
    question_text = merged_row.get("question", "")[:120] + "..." if len(merged_row.get("question", "")) > 120 else merged_row.get("question", "")
    
    for m in model_columns:
        val = merged_row.get(m, None)
        if pd.isna(val) or is_error(val):
            issues.append({
                "index": idx,
                "model": m,
                "column_type": "response",
                "issue_type": "missing" if pd.isna(val) else "error",
                "question": question_text,
                "value": str(val)[:100] + "..." if val and len(str(val)) > 100 else str(val) if val else None
            })
        
        # Check for reasoning in response
        if not pd.isna(val) and not is_error(val):
            has_reasoning_val = has_reasoning(val)
            reasoning_stats.append({
                "index": idx,
                "model": m,
                "has_reasoning": has_reasoning_val
            })
    
    # Check separate reasoning columns
    for r_col in reasoning_columns:
        val = merged_row.get(r_col, None)
        if pd.isna(val):
            issues.append({
                "index": idx,
                "model": r_col,
                "column_type": "reasoning",
                "issue_type": "missing",
                "question": question_text,
                "value": None
            })
        elif is_error(val):
            issues.append({
                "index": idx,
                "model": r_col,
                "column_type": "reasoning",
                "issue_type": "error",
                "question": question_text,
                "value": str(val)[:100] + "..." if val and len(str(val)) > 100 else str(val)
            })
        else:
            reasoning_stats.append({
                "index": idx,
                "model": r_col,
                "has_reasoning": True
            })

issues_df = pd.DataFrame(issues)
reasoning_df = pd.DataFrame(reasoning_stats)

# --- Reasoning Statistics ---
if not reasoning_df.empty:
    print("\n" + "="*60)
    print("Reasoning Availability Statistics:")
    print("="*60)
    reasoning_summary = reasoning_df.groupby("model").agg({
        "has_reasoning": ["sum", "count"]
    })
    reasoning_summary.columns = ["with_reasoning", "total"]
    reasoning_summary["percentage"] = (reasoning_summary["with_reasoning"] / reasoning_summary["total"] * 100).round(1)
    reasoning_summary = reasoning_summary.sort_values("with_reasoning", ascending=False)
    display(reasoning_summary)
    
    print(f"\nOverall: {reasoning_df['has_reasoning'].sum()} / {len(reasoning_df)} responses have reasoning ({reasoning_df['has_reasoning'].mean()*100:.1f}%)")

# --- summary ---
if issues_df.empty:
    print("\n✅ No missing or error responses found.")
else:
    print(f"⚠️ Found {len(issues_df)} issues across {issues_df['index'].nunique()} questions.")
    print("\n" + "="*60)
    print("Issue type breakdown:")
    print("="*60)
    print(issues_df["issue_type"].value_counts())
    
    # Breakdown by column type (response vs reasoning)
    if "column_type" in issues_df.columns:
        print("\n" + "="*60)
        print("Issues by Column Type (Response vs Reasoning):")
        print("="*60)
        column_type_breakdown = pd.crosstab(issues_df["column_type"], issues_df["issue_type"], margins=True)
        display(column_type_breakdown)
    
    # Group errors by model
    print("\n" + "="*60)
    print("Issues grouped by Model:")
    print("="*60)
    model_summary = issues_df.groupby("model").agg({
        "index": "count",
        "issue_type": lambda x: dict(x.value_counts())
    }).rename(columns={"index": "total_count", "issue_type": "breakdown"})
    model_summary = model_summary.sort_values("total_count", ascending=False)
    display(model_summary)
    
    # Group errors by error value/message (for actual errors, not missing)
    error_df = issues_df[issues_df["issue_type"] == "error"].copy()
    if not error_df.empty:
        print("\n" + "="*60)
        print("Errors grouped by Error Message:")
        print("="*60)
        # Extract error message pattern (first part before colon or first 50 chars)
        def extract_error_pattern(x):
            if not x:
                return "Unknown"
            x_str = str(x)
            if ":" in x_str:
                return x_str.split(":")[0].strip()
            return x_str[:50].strip()
        
        error_df["error_pattern"] = error_df["value"].apply(extract_error_pattern)
        error_grouped = error_df.groupby("error_pattern").agg({
            "index": ["count", lambda x: sorted(list(x))[:5]],
            "model": lambda x: sorted(list(x.unique()))[:5]
        })
        error_grouped.columns = ["count", "sample_indices", "sample_models"]
        error_grouped = error_grouped.sort_values("count", ascending=False)
        display(error_grouped)
    
    # Group by model and issue_type (cross-tabulation)
    print("\n" + "="*60)
    print("Issues grouped by Model and Issue Type (Cross-tabulation):")
    print("="*60)
    model_issue_crosstab = pd.crosstab(issues_df["model"], issues_df["issue_type"], margins=True)
    display(model_issue_crosstab)
    
    print("\n" + "="*60)
    print("First few problematic rows:")
    print("="*60)
    display(issues_df.head(20))


Found 0 reasoning columns: []

Reasoning Availability Statistics:


,with_reasoning,total,percentage
model,,,
accounts_fireworks_models_kimi-k2-thinking_response,20,20,100.0
openai_gpt-oss-120b_response,20,20,100.0
openai_gpt-oss-20b_response,20,20,100.0
nvidia_NVIDIA-Nemotron-Nano-9B-v2_response,20,20,100.0
gemini-2.5-flash-lite_response,0,20,0.0
grok-4-fast-reasoning_response,0,20,0.0
gpt-5-nano-2025-08-07_response,0,20,0.0
google_gemma-3n-E4B-it_response,0,20,0.0
meta-llama_Llama-3.2-11B-Vision-Instruct_response,0,20,0.0



Overall: 80 / 180 responses have reasoning (44.4%)

✅ No missing or error responses found.
